# Protein–Ligand Interaction Energetics — Colab Notebook

Four methods for getting protein–ligand energy out of a GROMACS trajectory:

| # | Method | What it gives you | Cost |
|---|--------|-------------------|------|
| 1 | **`mdrun -rerun`** | Short-range MM interaction energy (Coul-SR + LJ-SR) | seconds |
| 2 | **MM-GBSA / MM-PBSA** | End-state binding free-energy estimate | minutes–hours |
| 3 | **Per-residue decomposition** | Hotspot map of which residues drive binding | minutes–hours |
| 4 | **LIE-style split** | Ligand–protein vs ligand–water contributions | seconds |

**Required files** (CHARMM-GUI-style names — change in the *Configuration* cell if yours differ):
`forcefield.itp`, `PROA.itp`, `LIG.itp`, `topol-6.top`, `index-2.ndx`, `step7_10-2.tpr`, `fixed.xtc`

**Run order:** Setup → Install GROMACS → (optional) Install gmx_MMPBSA → Upload files → Config → any combination of Methods 1–4 → Summary → Download.


## 1. Detect environment & mount Google Drive

Your inputs live on Drive at `MyDrive/MD/Dtpa/dockhuman2`. This cell mounts Drive read/write, points `DRIVE_DIR` at that folder, and creates a fast **local** scratch directory (`/content/work`) where GROMACS will actually read/write — Drive over FUSE is too slow for `.xtc` rerun I/O. Outputs are copied back to Drive in the final cell.

In [ ]:
import os, sys, subprocess, shutil, textwrap, json
from pathlib import Path

try:
    import google.colab            # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Drive folder where your seven GROMACS files live ---
DRIVE_DIR = Path("/content/drive/MyDrive/MD/Dtpa/dockhuman2")

# --- Fast local scratch for GROMACS I/O ---
if IN_COLAB:
    WORKDIR = Path("/content/work")
else:
    WORKDIR = Path.cwd() / "work"

WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if not DRIVE_DIR.exists():
        raise FileNotFoundError(
            f"DRIVE_DIR not found: {DRIVE_DIR}\n"
            "Check the path in the cell above and re-run."
        )
    print("Drive mounted at /content/drive")
    print("DRIVE_DIR contents:")
    for p in sorted(DRIVE_DIR.iterdir()):
        if p.is_dir():
            kids = sorted(p.iterdir())
            print(f"  {p.name + '/':25s} <dir, {len(kids)} files>")
            for k in kids:
                print(f"    {k.name:23s} {k.stat().st_size/1e6:8.2f} MB")
        else:
            print(f"  {p.name:25s} {p.stat().st_size/1e6:8.2f} MB")

print("\nworkdir (local scratch):", WORKDIR)


## 2. Install GROMACS

We use [`condacolab`](https://github.com/conda-incubator/condacolab) to bring conda into Colab and pull GROMACS from `bioconda`. **The kernel will restart once after `condacolab.install()` — that is expected.** Re-run cells from the top after the restart.

If you already have a GROMACS install (running this notebook locally, not on Colab), skip to the next section.

In [ ]:
# Run this cell ONCE on Colab. It will restart the runtime.
if IN_COLAB:
    if shutil.which("gmx") is None:
        try:
            import condacolab
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab"])
            import condacolab
        if not condacolab.check():
            condacolab.install()
    else:
        print("gmx already on PATH:", shutil.which("gmx"))
else:
    print("local run — assuming `gmx` is already on PATH")
    print("found:", shutil.which("gmx"))


In [ ]:
# Run AFTER the runtime restart (Colab only). Installs the GROMACS binary.
if IN_COLAB and shutil.which("gmx") is None:
    !mamba install -y -c bioconda -c conda-forge gromacs 2>&1 | tail -5

print("gmx version:")
!gmx --version 2>&1 | head -10


## 3. (Optional) Install `gmx_MMPBSA`

Only needed for **Method 2** (MM-GBSA/PBSA) and **Method 3** (per-residue decomposition). Skip if you only want Methods 1 and 4. Takes ~5 minutes on Colab.

In [ ]:
INSTALL_MMPBSA = False   # ← flip to True if you want Methods 2 / 3

if INSTALL_MMPBSA and shutil.which("gmx_MMPBSA") is None:
    !mamba install -y -c conda-forge gmx_MMPBSA 2>&1 | tail -5

print("gmx_MMPBSA:", shutil.which("gmx_MMPBSA") or "NOT INSTALLED")


## 4. Stage input files from Drive

Symlinks the four top-level files **and** the entire `toppar/` directory from `DRIVE_DIR` into the local scratch dir. CHARMM-GUI's `topol.top` has `#include "toppar/..."` lines, so the relative folder must be present next to the `.top`.

In [ ]:
TOP_LEVEL = ["topol-6.top", "index-2.ndx", "step7_10-2.tpr", "fixed.xtc"]
TOPPAR_FILES = ["forcefield.itp", "PROA.itp", "LIG.itp"]   # inside toppar/

if IN_COLAB:
    missing = []

    # Top-level files: symlink directly into WORKDIR
    for f in TOP_LEVEL:
        src, dst = DRIVE_DIR / f, WORKDIR / f
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        if src.exists():
            dst.symlink_to(src)
        else:
            missing.append(f)

    # toppar/ folder: symlink the directory so all its includes resolve
    toppar_src = DRIVE_DIR / "toppar"
    toppar_dst = WORKDIR / "toppar"
    if toppar_dst.is_symlink() or toppar_dst.exists():
        if toppar_dst.is_symlink():
            toppar_dst.unlink()
        else:
            shutil.rmtree(toppar_dst)
    if toppar_src.exists():
        toppar_dst.symlink_to(toppar_src, target_is_directory=True)
        for f in TOPPAR_FILES:
            if not (toppar_src / f).exists():
                missing.append(f"toppar/{f}")
    else:
        missing.append("toppar/ (directory)")

    if missing:
        raise FileNotFoundError(
            f"Not found in {DRIVE_DIR}: {missing}\n"
            "Check filenames or copy them into that Drive folder."
        )

print("staged in workdir:")
for p in sorted(WORKDIR.iterdir()):
    target = f"  ->  {os.readlink(p)}" if p.is_symlink() else ""
    if p.is_dir():
        n = sum(1 for _ in p.iterdir())
        print(f"  {p.name:25s} <dir, {n} files>{target}")
    else:
        print(f"  {p.name:25s} {p.stat().st_size/1e6:8.2f} MB{target}")


## 5. Configuration — change here if your filenames or group names differ

In [ ]:
CFG = {
    "tpr":     "step7_10-2.tpr",
    "xtc":     "fixed.xtc",
    "ndx":     "index-2.ndx",
    "top":     "topol-6.top",
    "protein": "Protein",   # group names AS THEY APPEAR in the .ndx
    "ligand":  "LIG",
    "water":   "Water",
    "gmx":     "gmx",
}

OUT = WORKDIR / "energy_out"
OUT.mkdir(exist_ok=True)

def list_groups(ndx):
    out = []
    with open(ndx) as f:
        for line in f:
            line = line.strip()
            if line.startswith("[") and line.endswith("]"):
                out.append(line[1:-1].strip())
    return out

groups = list_groups(CFG["ndx"])
print("index groups available:")
for i, g in enumerate(groups):
    print(f"  [{i:2d}] {g}")

for key in ("protein", "ligand"):
    if CFG[key] not in groups:
        print(f"\n⚠️  {CFG[key]!r} not in index — change CFG[{key!r}] above to one of the names listed.")


## 6. Shared helpers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def run(cmd, *, stdin_text=None, cwd=None, check=True):
    """Run a shell command and stream output to the notebook."""
    print(f"$ {' '.join(map(str, cmd))}")
    proc = subprocess.run(list(map(str, cmd)),
                          input=stdin_text, text=True, cwd=cwd,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          check=False)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed (rc={proc.returncode})")
    return proc

def parse_xvg(path):
    times, rows, labels = [], [], []
    with open(path) as f:
        for line in f:
            if line.startswith(("@", "#")):
                if "legend" in line and line.split()[1].startswith("s"):
                    try:    labels.append(line.split('"')[1])
                    except IndexError: pass
                continue
            parts = line.split()
            if not parts: continue
            times.append(float(parts[0]))
            rows.append([float(x) for x in parts[1:]])
    return np.array(times), np.array(rows), labels

def summarize_and_plot(xvg, title):
    t, data, labels = parse_xvg(xvg)
    if not labels:
        labels = [f"col{i}" for i in range(data.shape[1])]
    df = pd.DataFrame({
        "term":  labels,
        "mean":  data.mean(axis=0),
        "sd":    data.std(axis=0),
        "sem":   data.std(axis=0) / np.sqrt(len(t)),
    })
    df.loc[len(df)] = ["TOTAL", df["mean"].sum(), np.nan,
                       np.sqrt((df["sem"]**2).sum())]
    print(f"\n=== {title} — {len(t)} frames, {t[0]:.0f}–{t[-1]:.0f} ps ===")
    display(df.style.format({"mean": "{:.2f}", "sd": "{:.2f}", "sem": "{:.2f}"}))

    fig, ax = plt.subplots(figsize=(9, 4))
    for i, lab in enumerate(labels):
        ax.plot(t, data[:, i], label=lab, lw=1)
    ax.set_xlabel("time (ps)"); ax.set_ylabel("energy (kJ/mol)")
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.tight_layout(); plt.show()
    return df


## 7. Method 1 — Short-range nonbonded MM interaction energy

Builds a minimal `.mdp` with `energygrps = Protein LIG`, recompiles the `.tpr`, reruns the trajectory through `mdrun -rerun`, and pulls `Coul-SR:Protein-LIG` and `LJ-SR:Protein-LIG` from the new `.edr`.

**This is not a binding free energy.** It is the gas-phase MM interaction (no solvation, no entropy). Useful for comparing poses or watching the interaction evolve in time.

In [ ]:
prot, lig = CFG["protein"], CFG["ligand"]

mdp_lines = [
    "integrator   = md",
    "nsteps       = 0",
    "dt           = 0.002",
    "nstenergy    = 1",
    "nstlog       = 1000",
    "cutoff-scheme = Verlet",
    "nstlist      = 20",
    "rlist        = 1.2",
    "coulombtype  = PME",
    "rcoulomb     = 1.2",
    "vdwtype      = cutoff",
    "vdw-modifier = force-switch",
    "rvdw-switch  = 1.0",
    "rvdw         = 1.2",
    "fourierspacing = 0.16",
    "pme-order    = 4",
    "constraints  = h-bonds",
    "constraint-algorithm = LINCS",
    f"energygrps   = {prot} {lig}",
]
mdp = OUT / "rerun.mdp"
mdp.write_text("\n".join(mdp_lines) + "\n")

tpr = OUT / "rerun.tpr"
edr = OUT / "rerun.edr"
xvg = OUT / "interaction_energy.xvg"

run([CFG["gmx"], "grompp", "-f", mdp, "-c", CFG["tpr"], "-p", CFG["top"],
     "-n", CFG["ndx"], "-o", tpr, "-maxwarn", "5"])
run([CFG["gmx"], "mdrun", "-s", tpr, "-rerun", CFG["xtc"],
     "-e", edr, "-g", OUT/"rerun.log", "-deffnm", str(OUT/"rerun")])

selectors = "\n".join([
    f"Coul-SR:{prot}-{lig}",
    f"LJ-SR:{prot}-{lig}",
    f"Coul-14:{prot}-{lig}",
    f"LJ-14:{prot}-{lig}",
    "",
])
run([CFG["gmx"], "energy", "-f", edr, "-o", xvg, "-sum", "no"],
    stdin_text=selectors)

df_method1 = summarize_and_plot(xvg, "Method 1 — Protein–LIG nonbonded (rerun)")


## 8. Method 4 — LIE-style ligand-environment split

Same machinery as Method 1, but with **three** energy groups (Protein, LIG, Water) so we can read out:
- LIG–Protein nonbonded (the "binding" contact)
- LIG–Water nonbonded (the cost of dehydration)

Big-magnitude LIG–Water with small LIG–Protein → ligand "wants" to be solvated → likely poor binder.

In [ ]:
prot, lig, wat = CFG["protein"], CFG["ligand"], CFG["water"]

mdp_lines = [
    "integrator   = md",
    "nsteps       = 0",
    "dt           = 0.002",
    "nstenergy    = 1",
    "nstlog       = 1000",
    "cutoff-scheme = Verlet",
    "nstlist      = 20",
    "rlist        = 1.2",
    "coulombtype  = PME",
    "rcoulomb     = 1.2",
    "vdwtype      = cutoff",
    "vdw-modifier = force-switch",
    "rvdw-switch  = 1.0",
    "rvdw         = 1.2",
    "fourierspacing = 0.16",
    "pme-order    = 4",
    "constraints  = h-bonds",
    "constraint-algorithm = LINCS",
    f"energygrps   = {prot} {lig} {wat}",
]
mdp = OUT / "lie.mdp"
mdp.write_text("\n".join(mdp_lines) + "\n")

tpr = OUT / "lie.tpr"
edr = OUT / "lie.edr"
xvg = OUT / "lie_decomposition.xvg"

run([CFG["gmx"], "grompp", "-f", mdp, "-c", CFG["tpr"], "-p", CFG["top"],
     "-n", CFG["ndx"], "-o", tpr, "-maxwarn", "5"])
run([CFG["gmx"], "mdrun", "-s", tpr, "-rerun", CFG["xtc"],
     "-e", edr, "-g", OUT/"lie.log", "-deffnm", str(OUT/"lie")])

selectors = "\n".join([
    f"Coul-SR:{lig}-{prot}",
    f"LJ-SR:{lig}-{prot}",
    f"Coul-SR:{lig}-{wat}",
    f"LJ-SR:{lig}-{wat}",
    "",
])
run([CFG["gmx"], "energy", "-f", edr, "-o", xvg, "-sum", "no"],
    stdin_text=selectors)

df_method4 = summarize_and_plot(xvg, "Method 4 — LIG–Protein vs LIG–Water (LIE-style)")


## 9. Method 2 — MM-GBSA / MM-PBSA binding free energy (`gmx_MMPBSA`)

End-state estimate of ΔG_bind. **Requires gmx_MMPBSA** — set `INSTALL_MMPBSA = True` in section 3 first.

GB is faster (start there), PB is more rigorous. Use a stride to avoid analyzing every single frame — every 10–20 ps of trajectory is plenty.

In [ ]:
MMPBSA_MODEL  = "gb"   # "gb", "pb", or "both"
MMPBSA_START  = 0
MMPBSA_END    = -1     # -1 = last frame
MMPBSA_STRIDE = 10
MMPBSA_NP     = 1      # MPI ranks

if shutil.which("gmx_MMPBSA") is None:
    raise RuntimeError("gmx_MMPBSA not installed — flip INSTALL_MMPBSA=True in section 3.")

# Build the &general / &gb / &pb input file as a list of lines
lines = [
    "&general",
    f"  startframe={MMPBSA_START}, endframe={MMPBSA_END}, interval={MMPBSA_STRIDE},",
    '  forcefields="leaprc.protein.ff14SB", PBRadii=3, temperature=310.0, verbose=2,',
    "/",
]
if MMPBSA_MODEL in ("gb", "both"):
    lines += ["&gb", "  igb=5, saltcon=0.150,", "/"]
if MMPBSA_MODEL in ("pb", "both"):
    lines += ["&pb", "  istrng=0.150, fillratio=4.0, radiopt=0, inp=2,", "/"]

inp = OUT / "mmpbsa.in"
inp.write_text("\n".join(lines) + "\n")

groups = list_groups(CFG["ndx"])
rec_id = groups.index(CFG["protein"])
lig_id = groups.index(CFG["ligand"])

cmd = ["mpirun", "-np", str(MMPBSA_NP), "gmx_MMPBSA", "MPI", "-O",
       "-i",  inp, "-cs", CFG["tpr"], "-ci", CFG["ndx"],
       "-cg", str(rec_id), str(lig_id),
       "-ct", CFG["xtc"], "-cp", CFG["top"],
       "-o",  OUT/"FINAL_RESULTS_MMPBSA.dat",
       "-eo", OUT/"FINAL_RESULTS_MMPBSA.csv"]
run(cmd, cwd=str(OUT))

csv = OUT / "FINAL_RESULTS_MMPBSA.csv"
if csv.exists():
    df_method2 = pd.read_csv(csv)
    display(df_method2.head())
    print("\nMean energies (kJ/mol):")
    display(df_method2.select_dtypes("number").agg(["mean", "std", "sem"]).T)


## 10. Method 3 — Per-residue decomposition

Same MM-GBSA run with `idecomp=2` so we get residue-by-residue contributions. Plotted as a bar chart of the most-contributing residues.

In [ ]:
DECOMP_STRIDE = 20

lines = [
    "&general",
    f"  startframe={MMPBSA_START}, endframe={MMPBSA_END}, interval={DECOMP_STRIDE},",
    '  forcefields="leaprc.protein.ff14SB", PBRadii=3, temperature=310.0, verbose=2,',
    "/",
    "&gb", "  igb=5, saltcon=0.150,", "/",
    "&decomp",
    '  idecomp=2, dec_verbose=0, print_res="within 6"',
    "/",
]
inp = OUT / "decomp.in"
inp.write_text("\n".join(lines) + "\n")

cmd = ["mpirun", "-np", str(MMPBSA_NP), "gmx_MMPBSA", "MPI", "-O",
       "-i",  inp, "-cs", CFG["tpr"], "-ci", CFG["ndx"],
       "-cg", str(rec_id), str(lig_id),
       "-ct", CFG["xtc"], "-cp", CFG["top"],
       "-o",  OUT/"FINAL_DECOMP_RESULTS.dat",
       "-eo", OUT/"FINAL_DECOMP_RESULTS.csv",
       "-do", OUT/"FINAL_DECOMP_MMPBSA.dat",
       "-deo", OUT/"FINAL_DECOMP_MMPBSA.csv"]
run(cmd, cwd=str(OUT))

dec_csv = OUT / "FINAL_DECOMP_MMPBSA.csv"
if dec_csv.exists():
    df_decomp = pd.read_csv(dec_csv)
    display(df_decomp.head(15))

    tot_col = next((c for c in df_decomp.columns
                    if "TOTAL" in c.upper() or "TDC" in c.upper()), None)
    res_col = next((c for c in df_decomp.columns
                    if "RES" in c.upper() or "RESIDUE" in c.upper()), df_decomp.columns[0])
    if tot_col is not None:
        top = df_decomp.reindex(df_decomp[tot_col].abs().sort_values(ascending=False).index).head(15)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(top[res_col].astype(str), top[tot_col],
               color=["#d33" if v > 0 else "#39c" for v in top[tot_col]])
        ax.axhline(0, color="k", lw=0.5)
        ax.set_ylabel("ΔG contribution (kJ/mol)")
        ax.set_title("Top 15 residues by |ΔG contribution|")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


## 11. Combined summary

In [ ]:
rows = []
if "df_method1" in globals():
    e = df_method1.set_index("term")["mean"]
    rows.append(("Method 1: nonbonded (rerun)",
                 f"Coul-SR + LJ-SR (TOTAL) = {e.get('TOTAL', float('nan')):.2f} kJ/mol"))
if "df_method4" in globals():
    e = df_method4["mean"].values[:-1]   # drop the TOTAL row
    rows.append(("Method 4: LIE LIG–Protein (Coul + LJ)", f"{e[0] + e[1]:.2f} kJ/mol"))
    rows.append(("Method 4: LIE LIG–Water   (Coul + LJ)", f"{e[2] + e[3]:.2f} kJ/mol"))
if "df_method2" in globals():
    rows.append(("Method 2: MM-GBSA ΔG_bind",
                 "see FINAL_RESULTS_MMPBSA.dat / .csv above"))

display(pd.DataFrame(rows, columns=["analysis", "result"]))


## 12. Save outputs back to Drive (and optional local download)

Two things happen here:
1. The whole `energy_out/` folder is copied to `DRIVE_DIR/energy_out/` so it survives a runtime timeout.
2. A zip is built and offered as a Colab download for convenience.

In [ ]:
import zipfile

# 1) Mirror outputs to Drive
if IN_COLAB:
    drive_out = DRIVE_DIR / "energy_out"
    drive_out.mkdir(exist_ok=True)
    for p in OUT.rglob("*"):
        if p.is_file():
            target = drive_out / p.relative_to(OUT)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(p, target)
    print(f"copied {sum(1 for _ in OUT.rglob('*') if _.is_file())} files -> {drive_out}")

# 2) Build a zip and offer download
zip_path = WORKDIR / "energy_out.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(WORKDIR))
print(f"\nzipped: {zip_path} ({zip_path.stat().st_size/1e6:.2f} MB)")

if IN_COLAB:
    # also copy the zip to Drive
    shutil.copy(zip_path, DRIVE_DIR / "energy_out.zip")
    print("copied zip ->", DRIVE_DIR / "energy_out.zip")
    from google.colab import files
    files.download(str(zip_path))


## Notes & gotchas

- **Match your production cutoffs.** The `.mdp` blocks above use CHARMM-GUI defaults (`rcoulomb=rvdw=1.2`, force-switch from 1.0, PME). If your `step6.x.mdp` used different cutoffs, edit the `mdp_lines` lists in Methods 1 and 4 — otherwise you are computing energies under slightly different physics than the trajectory was generated with.
- **Group naming is case-sensitive.** If section 5 reports your ligand group is called `Other` or `UNK` instead of `LIG`, update `CFG["ligand"]`.
- **Method 1 ≠ ΔG_bind.** It is the gas-phase MM interaction. For a free-energy estimate use Method 2.
- **Sampling.** Use `MMPBSA_STRIDE` to subsample — every 10–20 ps is usually fine, every frame is wasteful.
- **Entropy.** This notebook does *not* compute the −TΔS term. For a real ΔG you would add `entropy=2,` to the `&general` block (interaction entropy method), or run normal-mode analysis (slow).
- **Colab session limits.** Free Colab times out after ~12 hours and disconnects on inactivity. For long MM-PBSA runs, mount Drive and write `OUT` there, or use Colab Pro / a persistent VM.
